# Pandas Advanced Data Wrangling

**Module:** 1 — Foundations & Data Engineering  
**Week:** 1, Day 2  

## Learning Objectives
- Master DataFrame indexing: `.loc`, `.iloc`, boolean indexing
- GroupBy operations: aggregation, transformation, filtering
- Merge, join, and concatenate DataFrames
- Pivot tables and cross-tabulations
- Method chaining for clean, readable pipelines
- Handle missing data effectively

## Resources
- [Kaggle Learn: Pandas](https://www.kaggle.com/learn/pandas)
- [Pandas docs: 10 Minutes to Pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- [Real Python: Pandas GroupBy](https://realpython.com/pandas-groupby/)

In [ ]:
import pandas as pd
import numpy as np

# Setup shared utilities
import sys
sys.path.insert(0, '../..')
from shared.viz_utils import setup_style
setup_style()

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

---
## 1. Sample Dataset

We'll create a realistic sales dataset to work with.

In [ ]:
np.random.seed(42)
n = 500

sales = pd.DataFrame({
    "order_id": range(1000, 1000 + n),
    "date": pd.date_range("2024-01-01", periods=n, freq="D").repeat(1)[:n],
    "customer_id": np.random.choice([f"C{i:03d}" for i in range(1, 51)], n),
    "product": np.random.choice(["Laptop", "Mouse", "Keyboard", "Monitor", "Headset"], n,
                                 p=[0.15, 0.30, 0.25, 0.10, 0.20]),
    "quantity": np.random.randint(1, 10, n),
    "unit_price": np.random.choice([999.99, 29.99, 79.99, 399.99, 59.99], n,
                                    p=[0.15, 0.30, 0.25, 0.10, 0.20]),
    "region": np.random.choice(["North", "South", "East", "West"], n),
})

# Add some NaN values to practice handling missing data
sales.loc[sales.sample(20).index, "region"] = np.nan
sales.loc[sales.sample(10).index, "unit_price"] = np.nan

sales["revenue"] = sales["quantity"] * sales["unit_price"]

print(f"Shape: {sales.shape}")
print(f"\nDtypes:\n{sales.dtypes}")
print(f"\nMissing values:\n{sales.isnull().sum()}")
sales.head()

---
## 2. Indexing & Selection

Three main ways: `[]`, `.loc` (label-based), `.iloc` (position-based).

In [ ]:
# Boolean indexing — most common in data analysis
high_value = sales[sales["revenue"] > 1000]
print(f"High-value orders (>$1000): {len(high_value)}")

# Multiple conditions — use & (and), | (or), ~ (not)
laptop_north = sales[(sales["product"] == "Laptop") & (sales["region"] == "North")]
print(f"Laptop orders in North: {len(laptop_north)}")

# .loc — label-based (rows by condition, specific columns)
result = sales.loc[sales["product"] == "Monitor", ["customer_id", "quantity", "revenue"]]
print(f"\nMonitor orders (selected columns):")
result.head()

In [ ]:
# .query() — SQL-like syntax (cleaner for complex conditions)
result = sales.query("product == 'Laptop' and quantity >= 5 and region == 'East'")
print(f"Laptop orders, qty>=5, East region: {len(result)}")

# .isin() — check membership
tech = sales[sales["product"].isin(["Laptop", "Monitor"])]
print(f"Tech products (Laptop/Monitor): {len(tech)}")

# .between() — range check
mid_range = sales[sales["revenue"].between(100, 500)]
print(f"Mid-range revenue ($100-$500): {len(mid_range)}")

---
## 3. GroupBy — The Workhorse of Data Analysis

Split → Apply → Combine pattern.

In [ ]:
# Basic aggregation
product_summary = (
    sales
    .groupby("product")
    .agg(
        total_orders=("order_id", "count"),
        total_revenue=("revenue", "sum"),
        avg_quantity=("quantity", "mean"),
        avg_order_value=("revenue", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
)

print("Product Summary:")
product_summary

In [ ]:
# Multiple groupby keys
region_product = (
    sales
    .groupby(["region", "product"])
    ["revenue"]
    .sum()
    .unstack(fill_value=0)  # Pivot product to columns
)

print("Revenue by Region × Product:")
region_product

In [ ]:
# .transform() — apply function but keep original shape (great for adding columns)
sales["product_avg_revenue"] = (
    sales.groupby("product")["revenue"].transform("mean")
)
sales["above_product_avg"] = sales["revenue"] > sales["product_avg_revenue"]

print("Orders above their product's average revenue:")
sales[["product", "revenue", "product_avg_revenue", "above_product_avg"]].head(10)

In [ ]:
# .apply() — flexible but slower; use for complex group-level logic
def top_n_orders(group, n=3):
    """Return the top N orders by revenue for each group."""
    return group.nlargest(n, "revenue")

top_orders_by_region = (
    sales
    .dropna(subset=["region"])
    .groupby("region")
    .apply(top_n_orders, include_groups=False)
)

print("Top 3 orders per region:")
top_orders_by_region[["product", "revenue"]].head(12)

---
## 4. Merge & Join

Combining data from multiple sources — like SQL JOINs.

In [ ]:
# Create a customers table
customers = pd.DataFrame({
    "customer_id": [f"C{i:03d}" for i in range(1, 55)],  # 4 extra customers with no orders
    "name": [f"Customer_{i}" for i in range(1, 55)],
    "tier": np.random.choice(["Gold", "Silver", "Bronze"], 54),
    "signup_date": pd.date_range("2023-01-01", periods=54, freq="7D"),
})

# Product catalog with cost info
products = pd.DataFrame({
    "product": ["Laptop", "Mouse", "Keyboard", "Monitor", "Headset"],
    "category": ["Computing", "Peripheral", "Peripheral", "Display", "Audio"],
    "cost": [600.0, 10.0, 30.0, 200.0, 20.0],
})

# INNER JOIN — only matching rows
merged = sales.merge(customers, on="customer_id", how="inner")
print(f"Inner join: {len(merged)} rows")

# LEFT JOIN — keep all sales, add customer info
merged_left = sales.merge(customers, on="customer_id", how="left")
print(f"Left join: {len(merged_left)} rows")

# Merge with products to get cost/margin
with_cost = sales.merge(products, on="product", how="left")
with_cost["margin"] = with_cost["revenue"] - (with_cost["cost"] * with_cost["quantity"])

print(f"\nMargin by category:")
with_cost.groupby("category")["margin"].agg(["sum", "mean"]).round(2)

### Merge Types

| How | Keeps | SQL Equivalent |
|-----|-------|----------------|
| `inner` | Only matching rows | INNER JOIN |
| `left` | All left + matching right | LEFT JOIN |
| `right` | All right + matching left | RIGHT JOIN |
| `outer` | All rows from both | FULL OUTER JOIN |

---
## 5. Pivot Tables

Reshape data for summary views — like Excel pivot tables.

In [ ]:
# Pivot table: region × product → total revenue
pivot = sales.pivot_table(
    values="revenue",
    index="region",
    columns="product",
    aggfunc="sum",
    fill_value=0,
    margins=True,  # Add row/column totals
)

print("Revenue Pivot Table (Region × Product):")
pivot.round(2)

In [ ]:
# Cross-tabulation — frequency counts
xtab = pd.crosstab(sales["region"], sales["product"], margins=True)
print("Order Count (Region × Product):")
xtab

---
## 6. Method Chaining — Clean Pipelines

Chain operations for readable, step-by-step transformations.

In [ ]:
# Instead of assigning intermediate variables:
top_customers = (
    sales
    .dropna(subset=["revenue"])
    .merge(customers, on="customer_id", how="left")
    .groupby(["customer_id", "name", "tier"])
    .agg(
        total_revenue=("revenue", "sum"),
        order_count=("order_id", "count"),
        avg_order=("revenue", "mean"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
    .head(10)
    .assign(revenue_formatted=lambda df: df["total_revenue"].map("${:,.2f}".format))
)

print("Top 10 Customers by Revenue:")
top_customers[["customer_id", "name", "tier", "revenue_formatted", "order_count"]]

In [ ]:
# .pipe() — apply a custom function in the chain
def add_revenue_rank(df: pd.DataFrame) -> pd.DataFrame:
    """Add a rank column based on total revenue."""
    return df.assign(rank=df["total_revenue"].rank(ascending=False).astype(int))

ranked = (
    sales
    .dropna(subset=["revenue"])
    .groupby("customer_id")
    .agg(total_revenue=("revenue", "sum"))
    .reset_index()
    .pipe(add_revenue_rank)
    .sort_values("rank")
    .head(5)
)

print("Top 5 customers (with rank):")
ranked

---
## 7. Missing Data Strategies

In [ ]:
print(f"Missing values:\n{sales.isnull().sum()}")
print(f"\n% missing:\n{(sales.isnull().mean() * 100).round(1)}")

# Strategy 1: Drop rows
dropped = sales.dropna()
print(f"\nAfter dropna: {len(dropped)} rows (lost {len(sales) - len(dropped)})")

# Strategy 2: Fill with median/mode (better for numeric)
filled = sales.copy()
filled["unit_price"] = filled["unit_price"].fillna(filled["unit_price"].median())
filled["region"] = filled["region"].fillna(filled["region"].mode()[0])
filled["revenue"] = filled["quantity"] * filled["unit_price"]  # Recompute

print(f"After filling: {filled.isnull().sum().sum()} missing values")

---
## 8. Practice Exercises

### Exercise 1: Customer Cohort Analysis
Using the `sales` and `customers` DataFrames:
1. Merge them together
2. Group customers by their `tier` and calculate: total revenue, avg revenue per order, number of unique customers
3. Which tier has the highest average revenue per order?

In [ ]:
# Exercise 1: Your code here


### Exercise 2: Monthly Revenue Trend
1. Extract month from the date column
2. Calculate total revenue by month
3. Find the month with the highest revenue
4. Calculate month-over-month growth rate

In [ ]:
# Exercise 2: Your code here


### Exercise 3: Product Margin Analysis (Method Chaining)
Write a single method chain that:
1. Merges sales with products
2. Calculates margin per order
3. Groups by product and region
4. Computes total margin and margin percentage
5. Sorts by total margin descending
6. Shows the top 10 results

In [ ]:
# Exercise 3: Your code here


---
## Key Takeaways

1. **`.loc`** for label/condition-based selection, **`.iloc`** for position-based
2. **`.query()`** is cleaner than chained boolean conditions
3. **GroupBy** is split-apply-combine: `.agg()` for summaries, `.transform()` for same-shape results
4. **`.merge()`** = SQL JOIN. Always specify `how=` explicitly
5. **Method chaining** with `.pipe()` and `.assign()` keeps code readable
6. **Handle missing data** early — check with `.isnull().sum()`, then decide: drop, fill, or flag